# cap3 信号、目标与持仓逐季度交付审计

只读封存 f696 Qlib 特征与 cap3 自身账户；全部944日、16季度；只分析事前登记字段，季度为事后描述，不选新因子或训练。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import json,hashlib
from pathlib import Path
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/"research/stage1_cap3_signal_delivery_audit_20261030"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
source=root/plan["source_frame_card"];candidate=root/plan["cap3_card"]
assert sha(source/"verification_manifest.json")==plan["source_frame_manifest_sha256"]
assert sha(candidate/"verification_manifest.json")==plan["cap3_manifest_sha256"]
qlib.init(provider_uri=str(root/"data/qlib/stage1_domain_heads_20261018"),region="us",kernels=1)
src,src_receipt=load_verified_recorder_objects(source,expected_recorder_id=plan["source_frame_recorder_id"],artifacts=["inputs/frame"],expected_manifest_sha256=plan["source_frame_manifest_sha256"])
cap,cap_receipt=load_verified_recorder_objects(candidate,expected_recorder_id=plan["cap3_recorder_id"],artifacts=["paths/no_beta_cap_0"],expected_manifest_sha256=plan["cap3_manifest_sha256"])
frame=src["inputs/frame"].reset_index(drop=True);path=cap["paths/no_beta_cap_0"]
target=path["targets"].reset_index(drop=True);replay=path["replay"].reset_index(drop=True)
assert len(frame)==len(target)==len(replay)==plan["sessions"]
assert frame.trade_date.equals(target.trade_date) and frame.trade_date.equals(replay.trade_date)
assert frame.signal_date.equals(target.signal_date)
assert pd.to_datetime(frame.feature_available_at).lt(pd.to_datetime(target.decision_at)).all()
assert pd.to_datetime(frame.feature_available_at).dt.normalize().equals(pd.to_datetime(frame.signal_date).dt.normalize())
assert pd.to_datetime(frame.feature_available_at).dt.strftime("%H:%M").le("16:01").all()
assert pd.to_datetime(target.decision_at).dt.strftime("%H:%M").eq("09:46").all()
assert frame[list(plan["preselected_signal_fields"])].notna().all().all()
assert json.loads((candidate/"validation.json").read_text())["f696_five_output_parity_rows"]==944
lineage_path=root/plan["architecture_lineage"]
assert sha(lineage_path)==plan["architecture_lineage_sha256"]
lineage=json.loads(lineage_path.read_text())
assert lineage["saved_training_model"]["base_features"]==39 and lineage["saved_training_model"]["extra_features"]==0
joined=frame[["signal_date","trade_date","feature_available_at",*plan["preselected_signal_fields"]]].copy()
for name in ("soxl_weight","soxx_weight","cash_weight"):
    joined["accepted_"+name]=target[name].to_numpy(float)
for name in ("ending_soxl_weight","ending_soxx_weight","ending_cash_weight","risk_state","execute_open_target"):
    joined[name]=replay[name].to_numpy()
joined["year"]=joined.trade_date.dt.year
joined["quarter"]=joined.trade_date.dt.to_period("Q").astype(str)
rows=[]
for period,part in joined.groupby("quarter",sort=True):
    row={"period":period,"sessions":len(part),"first_trade":str(part.trade_date.min().date()),"last_trade":str(part.trade_date.max().date())}
    for name in plan["preselected_signal_fields"]:
        row["signal_"+name]=float(part[name].mean())
    for name in ("soxl_weight","soxx_weight","cash_weight"):
        row["accepted_"+name]=float(part["accepted_"+name].mean())
    for name in ("ending_soxl_weight","ending_soxx_weight","ending_cash_weight"):
        row[name]=float(part[name].mean())
    row["risk_state_nonzero_days"]=int(part.risk_state.ne(0).sum())
    row["execute_days"]=int(part.execute_open_target.sum())
    rows.append(row)
result=pd.DataFrame(rows)
exact=pd.read_csv(candidate/"interval_attribution.csv")
assert len(result)==len(exact)==16 and result.period.tolist()==exact.period.tolist()
for name in ("relative_wealth_vs_soxx","candidate_return_pct","soxx_return_pct","candidate_vs_parent_relative_wealth"):
    result[name]=exact[name].to_numpy(float)
result.to_csv(card/"quarter_signal_delivery.csv",index=False)
focus=result.loc[result.period.isin(["2024Q3","2026Q2","2026Q3"])]
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"status":"descriptive_signal_delivery_no_candidate",
       "sessions":len(frame),"quarters":len(result),"source_readback_entries":{"f696_frame":len(src_receipt),"cap3_account":len(cap_receipt)},
       "same_day_minute_in_f696":False,"prior_day_minute_aggregates_used_by_f696":False,"feature_clock":"prior-session 16:01 or earlier, before cap3 09:46", 
       "new_model_fits":0,"new_native_accounts":0,"candidate_feature_selected":False,
       "focus":focus[["period","sessions","signal_alpha_probability","signal_tail_probability","signal_cash_weight","signal_soxx_ret20","signal_soxx_vol20","signal_minute_soxx_rebound_from_low","accepted_soxl_weight","accepted_soxx_weight","accepted_cash_weight","relative_wealth_vs_soxx","candidate_vs_parent_relative_wealth"]].to_dict("records"),
       "limits":["Quarter labels are hindsight outcomes; no causal signal or tradable switch is identified.","Existing minute_* frame values are prior-session summaries, not intraday event features.","The original b472 action model was not refitted for cap3; recovery eligibility changed mechanically."]}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"status":audit["status"],"sessions":len(frame),"focus":audit["focus"]},ensure_ascii=False))

## 中文结论

封存944日f696五输出、cap3目标/实际持仓逐行对齐；全16季度见quarter_signal_delivery.csv。SOXX 0bp财富与风险六门全败，沿用原账户卡。2026Q2事前SOXX20日收益均值+21.05%，f696原SOXL提案29.44%，cap3接受29.25%，相对SOXX财富1.199；2026Q3相应收益−3.16%、60日回撤−17.32%，原提案22.13%、接受22.04%、末端21.88%，相对SOXX0.869。2024Q3原提案13.50%仍相对SOXX0.945。来源与09:46时钟通过，minute_*是前一会话16:01已完成的研究摘要，保存的f696模型只有39个基础输入、0个额外分钟输入，连这些前日分钟摘要也没用；旧b472开盘特征同样没有当日逐分钟字段。季度是事后解释，不选阈值或特征；单个特征因果影响unknown。零新拟合/账户，不晋升。下一步仅在找到与旧路线不同的决策前信息机制后冻结新接收器。

NAS边界：f696源已封存并限定读回；cap3新Recorder仅本地哈希读回，NAS上传收据与限定恢复仍缺，本地对象保留。